In [ ]:
import pandas as pd
import chromadb
from src.chunker import (
    chunk_markdown,
    chunk_csv,
    _fmt_appendix1, _meta_appendix1,
    _fmt_appendix2, _meta_appendix2,
    _fmt_appendix3, _meta_appendix3,
    _fmt_table1, _meta_table1,
    _fmt_table2, _meta_table2,
    _fmt_table4, _meta_table4,
)
from src.embedder import (
    store,
    embed,
)

from src.router import(
    route,
)
from src.retriever import(
    retrieve,
)

In [ ]:
df_appendix1 = pd.read_csv("rag_data/appendix1_questionnairemasterandscoring.csv", sep=";", encoding="latin-1", dtype={"no": str})
df_appendix1["answer"] = df_appendix1["answer"].astype(str).str.replace("?", "≤", regex=False)
df_appendix2 = pd.read_csv('rag_data/appendix2_listofgreenbuildingelements.csv', sep=';', dtype={'no': str})
df_appendix3 = pd.read_csv('rag_data/appendix3_listanddescriptionofsmartbuildingrequirements.csv', sep='\t', dtype={'no': str})
df_appendix3['description'] = df_appendix3['description'].str.replace('CO?', 'CO₂', regex=False)

df_table1 = pd.read_csv('rag_data/table1_nationalcoordinators.csv', sep=';', encoding='utf-8', dtype={'no': str})
df_table2 = pd.read_csv('rag_data/table2_categoriesusedandweighting.csv', sep=';', encoding='utf-8', dtype={'no': str})
df_table4 = pd.read_csv('rag_data/table4_greenhousegasemissionsources.csv', sep=';', encoding='utf-8', dtype={'no': str})

In [ ]:
sources = {}
sources["pdf"] = chunk_markdown("rag_data/guidelines_markdown.md")
sources["csv_appendix1"] = chunk_csv(df_appendix1, "no", source="csv_appendix1", chunk_type="question", format_fn=_fmt_appendix1, metadata_fn=_meta_appendix1)
sources["csv_appendix2"] = chunk_csv(df_appendix2, "element_category", source="csv_appendix2", chunk_type="reference", format_fn=_fmt_appendix2, metadata_fn=_meta_appendix2)
sources["csv_appendix3"] = chunk_csv(df_appendix3, "field_code", source="csv_appendix3", chunk_type="reference", format_fn=_fmt_appendix3, metadata_fn=_meta_appendix3)
sources["csv_table1"] = chunk_csv(df_table1, "country", source="csv_table1", chunk_type="reference", format_fn=_fmt_table1, metadata_fn=_meta_table1)
sources["csv_table2"] = chunk_csv(df_table2, "category", source="csv_table2", chunk_type="reference", format_fn=_fmt_table2, metadata_fn=_meta_table2)
sources["csv_table4"] = chunk_csv(df_table4, "scope", source="csv_table4", chunk_type="reference", format_fn=_fmt_table4, metadata_fn=_meta_table4)

In [ ]:
store(sources)

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("greenmetric_v05")
print(f"Total chunks stored: {collection.count()}")
print(collection.peek(3))

In [ ]:
results = collection.query(
    query_embeddings=embed(["Does the questionnaire include questions about energy consumption?"]),
    n_results=2
)
results

In [ ]:
# Function testing for the router + retriever with 5 test queries.
test_queries = [
    ["Why is sustainability so important? And why do we even need rankings?", "Expected source: none", "Expected query type: lookup"],
    ["Kampusku ada lahan buat pertanian yang disewakan untuk menunjang keuangan kampus, apakah bisa dianggap hutan kampus? Dan seberapa besar lahan hijau yang diperlukan untuk mendapatkan nilai maksimum untuk kriteria ini?", "Expected source: both_pdf_and_csv_appendix1", "Expected query type: lookup"],
    ["Which category has the most questions in it?", "Expected source: csv_appendix1", "Expected query type: aggregate"],
    ["How many options does indicator 5.1 have?", "Expected source: csv_appendix1", "Expected query type: lookup"],
    ["Which category has the longest explanation?", "Expected source: pdf", "Expected query type: aggregate"],
]
for query, expected_source, expected_query_type in test_queries:
    route_result = route(query)
    retrieval_result = retrieve(query, route_result)
    print(f"Query: {query}")
    print(f"Route result: {route_result}")
    print(f"{expected_source}")
    print(f"{expected_query_type}")
    print(f"Retrieved: {len(retrieval_result)} chunks")
    if retrieval_result:
        for r in retrieval_result[:2]:
            print(f"  dist={r['distance']:.4f}  src={r['metadata']['source']}  type={r['metadata']['chunk_type']}")
        if len(retrieval_result) > 2:
            print(f"  ... and {len(retrieval_result) - 2} more")
    print("-" * 50)

In [ ]:
from src.router import route
from src.retriever import retrieve
from src.generator import generate

for query in [
    "What is the maximum score for water conservation program implementation?",
    "Which category has the most questions in it?"
]:
    route_result = route(query)
    chunks = retrieve(query, route_result)
    answer = generate(query, chunks, query_type=route_result["query_type"])
    print(f"Q: {query}")
    print(f"Route: {route_result}")
    print(f"Chunks retrieved: {len(chunks)}")
    print(f"A: {answer}")
    print("-" * 60)

In [ ]:
from src.pipeline import ask

result1 = ask("What are the questions that are in the first category?")
print(f"Turn 1: {result1['answer'][:100]}...")
print(f"History length: {len(result1['conversation_history'])}")

result2 = ask(
    "And which question has the lowest non 0 score for that category?",
    conversation_history=result1["conversation_history"]
)
print(f"Turn 2: {result2['answer'][:100]}...")
print(f"History length: {len(result2['conversation_history'])}")

In [ ]:
print(result1['answer'])

In [ ]:
print(result2['answer'])

In [ ]:
from src.router import route
from src.retriever import retrieve

r = route("What is UIGM criteria 2.5 about and how many options does it have?")
chunks = retrieve("What is UIGM criteria 2.5 about and how many options does it have?", r)
for c in chunks:
    print(c['metadata'].get('question_no'), c['distance'])

In [ ]:
import gradio
print(gradio.__version__)
import inspect
print(inspect.signature(gradio.Chatbot.__init__))

In [4]:
from src.evaluate import run_evaluation
result = run_evaluation()
# Pretty print
print(f"Faithfulness:      {result['overall']['faithfulness']:.2f}")
print(f"Context Recall:    {result['overall']['context_recall']:.2f}")
print(f"Response Relevancy:{result['overall']['response_relevancy']:.2f}")
print(f"Answer Correctness:{result['overall']['answer_correctness']:.2f}")
print(f"Router Accuracy:   {result['router_accuracy']:.1%}")
print(f"None cases:        {result['none_cases']:.0%}")
print()
print(result['by_source'])
print()
print(result['weakest'])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[1/40] Kampusku ada lahan buat pertanian yang disewakan untuk menunjang keuangan kampus...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


[40/40] What is the maximum allowable greenhouse gas (CO2) emission tonnage for a univer...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Faithfulness:      0.00
Context Recall:    0.00
Response Relevancy:0.00
Answer Correctness:0.00
Router Accuracy:   77.5%
None cases:        75%

                    source  faithfulness  context_recall  response_relevancy  \
0   both_pdf_csv_appendix1           0.0             0.0                 0.0   
1   both_pdf_csv_appendix2           NaN             NaN                 NaN   
2   both_pdf_csv_appendix3           NaN             NaN                 NaN   
3      both_pdf_csv_table1           NaN             NaN                 NaN   
4      both_pdf_csv_table2           NaN             NaN                 NaN   
5      both_pdf_csv_table4           NaN             NaN                 NaN   
6            csv_appendix1           NaN             NaN                 NaN   
7            csv_appendix2           NaN             NaN                 NaN   
8            csv_appendix3           NaN             NaN                 NaN   
9               csv_table1           NaN             Na

In [1]:
from src.evaluate import run_evaluation
result = run_evaluation('test_cases/test_cases_5.xlsx')
print(result['overall'])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model: paraphrase-multilingual-MiniLM-L12-v2
Embedding dimension: 384


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[1/5] Kampusku ada lahan buat pertanian yang disewakan untuk menunjang keuangan kampus...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[5/5] Pedoman utama menyebutkan "certified green building status" sebagai salah satu c...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'faithfulness': np.float64(0.4683333333333334), 'context_recall': np.float64(0.5), 'answer_correctness': np.float64(0.44105995610704285)}


In [ ]:
from src.evaluate import run_evaluation
result = run_evaluation('test_cases/test_cases_5.xlsx')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[1/5] Kampusku ada lahan buat pertanian yang disewakan untuk menunjang keuangan kampus...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]